# KOA early-grade collapse — revision experiments (BPEX-106665)

This notebook runs the additional experiments requested in the major revision, using the **unchanged training protocol**
of the original study (the training code in `koa_core.py` is copied verbatim from the notebook that produced the
original three-seed results; only export, exclusion and bookkeeping utilities were added).

**Experimental blocks (run in this priority order):**

| Tier | Purpose | Arms × seeds | New runs |
|---|---|---|---|
| A | Complete the 2×2 imbalance factorial at the original seeds (42, 7, 123) | WCE-only, sampler-only (3 seeds each), CE (seeds 7, 123) | 8 |
| B | Extend factorial and ordinal model to five seeds (2024, 2025) | CE, WCE-only, sampler-only, WCE+sampler, moderate ordinal | 10 |
| C | Retrain key models on a pHash-cleaned train/val split (test unchanged) | WCE+sampler, moderate ordinal × 3 seeds | 6 |
| D | Ordinal-weight sensitivity with blend (0.10) and margin (1.0) fixed | λ = 0.05/0.02 and 0.35/0.20 × 3 seeds | 6 |

Already-completed original runs (WCE+sampler and moderate ordinal at seeds 42/7/123, CE at seed 42) are reused and are not retrained.

**How to run on Kaggle**
1. Accelerator: **GPU T4 ×2** (two runs are trained in parallel, one per GPU). Internet: **ON** (ImageNet weights).
2. Add input: *Knee Osteoarthritis Dataset with Severity*.
3. First time only: set `SMOKE_TEST = True`, run all cells interactively (~10 min), check that the smoke test passes.
4. Set `SMOKE_TEST = False` and use **Save Version → Save & Run All (Commit)**. A session stops starting new runs
   before the 12-hour limit, so every started run finishes and is saved.
5. For the next session: add the previous version's **output** as an input and commit again. Completed runs are
   detected through their `DONE.json` marker and skipped.
6. Download `koa_revision_results.zip` from the output (model checkpoints are excluded from the zip).

In [ ]:
# ======================= SETTINGS =======================
import time, json, os, sys, shutil, subprocess, glob
from pathlib import Path

SESSION_START = time.time()
MAX_SESSION_HOURS = 11.0        # Kaggle hard limit is 12 h; keep a safety margin
EST_RUN_HOURS = 1.9             # conservative per-run estimate (observed 1.0-1.6 h on a single T4); adapts after each run
SMOKE_TEST = False              # True: one short end-to-end check (1 epoch) instead of the real queue
ENABLED_TIERS = ["A", "B", "C", "D"]
ORIGINAL_SEEDS = [42, 7, 123]
NEW_SEEDS = [2024, 2025]

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
OUT_DIR = WORK / "koa_revision_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = ""                  # leave empty for auto-detection
print("Output:", OUT_DIR)

In [ ]:
%%writefile koa_core.py
# koa_core.py — generated by the revision notebook. Do not edit by hand.
# Sections marked "verbatim" reproduce the original training code unchanged.
# ---- verbatim from original notebook: 1) IMPORTS AND ENV ----
import os, re, time, math, random, json, copy, warnings

from pathlib import Path

from collections import Counter, defaultdict

import numpy as np

import pandas as pd

from PIL import Image, ImageOps

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset

from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    mean_absolute_error, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # For reproducibility. Turning benchmark off can slow training.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# ---- verbatim from original notebook: 2) DATASET DISCOVERY ----
def find_data_root(user_root: str = ""):
    candidates = []
    if user_root:
        candidates.append(Path(user_root))

    # Common Kaggle / Colab / local candidates
    candidates += [
        Path("/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity"),
        Path("/kaggle/input/knee-osteoarthritis-dataset-with-severity/kneeOA"),
        Path("/kaggle/input/knee-osteoarthritis-dataset-with-severity/Knee Osteoarthritis Dataset with Severity"),
        Path("/kaggle/input"),
        Path("/content"),
        Path("/mnt/data"),
        Path("."),
    ]

    split_names = [
        ("train", "val", "test"),
        ("train", "valid", "test"),
        ("train", "validation", "test"),
        ("Training", "Validation", "Testing"),
    ]

    def has_splits(p: Path):
        if not p.exists() or not p.is_dir():
            return None
        children = {c.name.lower(): c for c in p.iterdir() if c.is_dir()}
        for tr, va, te in split_names:
            if tr.lower() in children and va.lower() in children and te.lower() in children:
                return {
                    "train": children[tr.lower()],
                    "val": children[va.lower()],
                    "test": children[te.lower()],
                }
        return None

    # Direct candidates
    for c in candidates:
        r = has_splits(c)
        if r:
            return c, r

    # Search two levels deep in Kaggle input
    for base in [Path("/kaggle/input"), Path("/content"), Path(".")]:
        if base.exists():
            for p in list(base.glob("*")) + list(base.glob("*/*")):
                r = has_splits(p)
                if r:
                    return p, r

    raise FileNotFoundError(
        "Dataset root not found. Set CONFIG['DATA_ROOT'] to a folder containing train/val/test."
    )


# ---- verbatim from original notebook: 3) DATASET CLASS ----
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def parse_grade_from_folder(folder_name: str):
    # Works for: "0", "1", "Grade 2", "2Moderate", "3_severe", etc.
    m = re.search(r"([0-4])", folder_name)
    if m:
        return int(m.group(1))
    return None

class KneeOADataset(Dataset):
    def __init__(self, split_dir, transform=None, allowed_labels=None):
        self.split_dir = Path(split_dir)
        self.transform = transform
        self.samples = []
        self.allowed_labels = set(allowed_labels) if allowed_labels is not None else None

        for class_dir in sorted([p for p in self.split_dir.iterdir() if p.is_dir()]):
            y = parse_grade_from_folder(class_dir.name)
            if y is None:
                continue
            if self.allowed_labels is not None and y not in self.allowed_labels:
                continue
            for img_path in class_dir.rglob("*"):
                if img_path.suffix.lower() in IMG_EXTS:
                    self.samples.append((img_path, y))

        if not self.samples:
            raise RuntimeError(f"No image samples found in {self.split_dir}")

        self.samples = sorted(self.samples, key=lambda x: str(x[0]))
        self.labels = np.array([y for _, y in self.samples], dtype=np.int64)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(y, dtype=torch.long), torch.tensor(idx, dtype=torch.long)

def summarize_split(name, ds):
    cnt = Counter(ds.labels.tolist())
    row = {"split": name, "n": len(ds)}
    for g in range(5):
        row[f"grade_{g}"] = cnt.get(g, 0)
    return row


# ---- verbatim from original notebook: 4) TRANSFORMS AND LOADERS ----
IMAGENET_MEAN = [0.485, 0.456, 0.406]

IMAGENET_STD  = [0.229, 0.224, 0.225]

class AutoContrast:
    def __call__(self, img):
        return ImageOps.autocontrast(img)

class CropNonEmpty:
    """Crop dark/empty borders in X-ray images before resizing.

    This is deliberately conservative. If no reliable mask is found, the image is returned unchanged.
    """
    def __init__(self, threshold=8, pad_ratio=0.03):
        self.threshold = threshold
        self.pad_ratio = pad_ratio

    def __call__(self, img):
        arr = np.asarray(img.convert("L"))
        mask = arr > self.threshold
        if mask.sum() < 50:
            return img
        ys, xs = np.where(mask)
        x0, x1 = xs.min(), xs.max() + 1
        y0, y1 = ys.min(), ys.max() + 1
        h, w = arr.shape
        pad = int(max(h, w) * self.pad_ratio)
        x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
        x1 = min(w, x1 + pad); y1 = min(h, y1 + pad)
        if (x1 - x0) < 0.40 * w or (y1 - y0) < 0.40 * h:
            return img
        return img.crop((x0, y0, x1, y1))

def build_transforms(img_size=224, use_roi_autocrop=True):
    pre = []
    if use_roi_autocrop:
        pre.append(CropNonEmpty(threshold=8, pad_ratio=0.03))
    pre.append(AutoContrast())

    # v3: high-resolution, conservative augmentation.
    # KL grading depends on joint-space details; avoid aggressive crop/rotation.
    train_tf = transforms.Compose(pre + [
        transforms.Resize(int(img_size * 1.08)),
        transforms.RandomResizedCrop(img_size, scale=(0.94, 1.00), ratio=(0.97, 1.03)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=3),
        transforms.ColorJitter(brightness=0.04, contrast=0.08),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    eval_tf = transforms.Compose(pre + [
        transforms.Resize(int(img_size * 1.08)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_tf, eval_tf

def class_counts_from_dataset(ds, num_classes=5):
    cnt = np.bincount(ds.labels, minlength=num_classes)
    return cnt

def make_sampler(ds, sampler_power=0.65, grade1_extra=1.35):
    counts = class_counts_from_dataset(ds)
    inv = 1.0 / np.maximum(counts, 1)
    weights_per_class = inv ** sampler_power
    weights_per_class[1] *= grade1_extra
    sample_weights = weights_per_class[ds.labels]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )

def build_loaders(config, allowed_labels=None, shuffle_train=True):
    train_tf, eval_tf = build_transforms(
        config["IMG_SIZE"],
        use_roi_autocrop=config.get("USE_ROI_AUTOCROP", True),
    )
    train_ds = KneeOADataset(SPLITS["train"], transform=train_tf, allowed_labels=allowed_labels)
    val_ds = KneeOADataset(SPLITS["val"], transform=eval_tf, allowed_labels=allowed_labels)
    test_ds = KneeOADataset(SPLITS["test"], transform=eval_tf, allowed_labels=allowed_labels)

    if config.get("USE_WEIGHTED_SAMPLER", False) and allowed_labels is None:
        sampler = make_sampler(
            train_ds,
            sampler_power=config["SAMPLER_POWER"],
            grade1_extra=config["GRADE1_EXTRA_WEIGHT"],
        )
        train_loader = DataLoader(
            train_ds,
            batch_size=config["BATCH_SIZE"],
            sampler=sampler,
            num_workers=config["NUM_WORKERS"],
            pin_memory=True,
            drop_last=True,
        )
    else:
        train_loader = DataLoader(
            train_ds,
            batch_size=config["BATCH_SIZE"],
            shuffle=shuffle_train,
            num_workers=config["NUM_WORKERS"],
            pin_memory=True,
            drop_last=True,
        )

    val_loader = DataLoader(
        val_ds,
        batch_size=max(1, config["BATCH_SIZE"] * 2),
        shuffle=False,
        num_workers=config["NUM_WORKERS"],
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=max(1, config["BATCH_SIZE"] * 2),
        shuffle=False,
        num_workers=config["NUM_WORKERS"],
        pin_memory=True,
    )
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader


# ---- verbatim from original notebook: 5) METRICS ----
def safe_div(a, b):
    return float(a) / float(b) if b else 0.0

def compute_metrics(y_true, y_pred, probs=None, prefix=""):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    labels = np.arange(5)

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )

    out = {
        prefix + "acc": acc,
        prefix + "bacc": bacc,
        prefix + "mae": mae,
        prefix + "qwk": qwk,
        prefix + "macro_f1": float(np.mean(f1)),
        prefix + "g1_recall": float(rec[1]),
        prefix + "g1_precision": float(prec[1]),
        prefix + "g1_f1": float(f1[1]),
    }
    for g in labels:
        out[prefix + f"recall_g{g}"] = float(rec[g])
        out[prefix + f"precision_g{g}"] = float(prec[g])
        out[prefix + f"f1_g{g}"] = float(f1[g])
        out[prefix + f"support_g{g}"] = int(support[g])
    return out

def val_selection_score(metrics, config):
    return (
        config["VAL_SCORE_ACC_W"] * metrics["acc"]
        + config["VAL_SCORE_QWK_W"] * metrics["qwk"]
        + config["VAL_SCORE_G1_W"] * metrics["g1_recall"]
    )

def print_metrics(m, title="metrics"):
    keys = ["acc", "qwk", "mae", "bacc", "macro_f1", "g1_recall", "g1_precision", "g1_f1"]
    print("\n" + title)
    for k in keys:
        if k in m:
            print(f"  {k:14s}: {m[k]:.4f}")

def per_class_table(y_true, y_pred):
    labels = np.arange(5)
    prec, rec, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    rows = []
    for g in labels:
        main_conf = ""
        errors = cm[g].copy()
        errors[g] = 0
        if errors.sum() > 0:
            j = int(np.argmax(errors))
            main_conf = f"Grade {j} ({errors[j]} / {errors.sum()} errors)"
        rows.append({
            "grade": g,
            "n": int(support[g]),
            "recall": rec[g],
            "precision": prec[g],
            "f1": f1[g],
            "main_confusion": main_conf,
        })
    return pd.DataFrame(rows)

def bootstrap_ci(y_true, y_pred, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    vals = defaultdict(list)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        m = compute_metrics(y_true[idx], y_pred[idx])
        for k in ["acc", "qwk", "mae", "g1_recall", "macro_f1"]:
            vals[k].append(m[k])
    rows = []
    for k, arr in vals.items():
        arr = np.asarray(arr)
        rows.append({
            "metric": k,
            "mean": float(np.mean(arr)),
            "ci_low": float(np.percentile(arr, 2.5)),
            "ci_high": float(np.percentile(arr, 97.5)),
        })
    return pd.DataFrame(rows)


# ---- verbatim from original notebook: 6) LOSSES ----
def class_balanced_weights(counts, beta=0.999):
    counts = np.asarray(counts, dtype=np.float64)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / np.maximum(effective_num, 1e-12)
    weights = weights / weights.sum() * len(counts)
    return weights.astype(np.float32)

def inverse_sqrt_weights(counts):
    counts = np.asarray(counts, dtype=np.float64)
    w = 1.0 / np.sqrt(np.maximum(counts, 1))
    w = w / w.sum() * len(counts)
    return w.astype(np.float32)

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=1.5, label_smoothing=0.0):
        super().__init__()
        self.register_buffer("weight", None if weight is None else torch.tensor(weight, dtype=torch.float32))
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, target):
        weight = self.weight.to(logits.device) if self.weight is not None else None
        ce = F.cross_entropy(
            logits, target, weight=weight, reduction="none",
            label_smoothing=self.label_smoothing
        )
        pt = torch.exp(-ce.detach())
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()

def cumulative_targets(labels, num_classes=5):
    # q_t = 1[y > t], t=0..K-2
    thresholds = torch.arange(num_classes - 1, device=labels.device).view(1, -1)
    return (labels.view(-1, 1) > thresholds).float()

def coral_bce_loss(coral_logits, labels, pos_weight=None, num_classes=5):
    q = cumulative_targets(labels, num_classes=num_classes)
    if pos_weight is not None:
        pos_weight = pos_weight.to(coral_logits.device)
    return F.binary_cross_entropy_with_logits(coral_logits, q, pos_weight=pos_weight)

def threshold_pos_weights(labels, num_classes=5):
    y = torch.tensor(labels, dtype=torch.long)
    q = cumulative_targets(y, num_classes=num_classes)
    pos = q.sum(0)
    neg = q.shape[0] - pos
    return (neg / torch.clamp(pos, min=1.0)).float()

def pairwise_margin_ranking_loss(scores, labels, margin=1.0):
    # scores: larger means more severe
    labels = labels.view(-1)
    scores = scores.view(-1)
    ydiff = labels.view(-1,1) - labels.view(1,-1)
    mask = ydiff > 0
    if mask.sum() == 0:
        return scores.sum() * 0.0
    sdiff = scores.view(-1,1) - scores.view(1,-1)
    return F.relu(margin - sdiff)[mask].mean()

def build_ce_criterion(config, train_labels):
    counts = np.bincount(train_labels, minlength=5)
    mode = config["LOSS_MODE"]

    if mode == "ce":
        weight = None
        return nn.CrossEntropyLoss(label_smoothing=config["LABEL_SMOOTHING"]), weight

    if mode == "weighted_ce":
        weight = inverse_sqrt_weights(counts)
        return nn.CrossEntropyLoss(
            weight=torch.tensor(weight, dtype=torch.float32),
            label_smoothing=config["LABEL_SMOOTHING"],
        ), weight

    if mode == "focal":
        weight = inverse_sqrt_weights(counts)
        return FocalLoss(weight=weight, gamma=config["FOCAL_GAMMA"], label_smoothing=config["LABEL_SMOOTHING"]), weight

    if mode == "cb_focal":
        weight = class_balanced_weights(counts, beta=config["CB_BETA"])
        # Extra emphasis for Grade 1, the reviewer-critical failure class.
        weight[1] *= 1.20
        weight = weight / weight.sum() * len(weight)
        return FocalLoss(weight=weight, gamma=config["FOCAL_GAMMA"], label_smoothing=config["LABEL_SMOOTHING"]), weight

    raise ValueError(f"Unknown LOSS_MODE: {mode}")

def hybrid_loss(outputs, labels, ce_criterion, config, coral_pos_weight=None):
    cls_logits = outputs["cls_logits"]
    coral_logits = outputs["coral_logits"]
    ordinal_score = outputs["ordinal_score"]

    ce = ce_criterion(cls_logits, labels)
    cor = coral_bce_loss(coral_logits, labels, pos_weight=coral_pos_weight)
    rank = pairwise_margin_ranking_loss(
        ordinal_score, labels, margin=config["PAIRWISE_MARGIN"]
    )

    loss = (
        config["CE_WEIGHT"] * ce
        + config["ORDINAL_WEIGHT"] * cor
        + config["RANK_WEIGHT"] * rank
    )
    return loss, {"loss_ce": float(ce.detach().cpu()), "loss_coral": float(cor.detach().cpu()), "loss_rank": float(rank.detach().cpu())}


# ---- verbatim from original notebook: 7) MODEL ----
class SEBlock(nn.Module):
    def __init__(self, dim, reduction=16):
        super().__init__()
        hidden = max(dim // reduction, 16)
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.net(x)

def _safe_weights(weight_enum, pretrained=True):
    if not pretrained:
        return None
    try:
        return weight_enum.DEFAULT
    except Exception:
        return None

def create_backbone(name="efficientnet_b3", pretrained=True):
    name = name.lower()
    try:
        if name == "efficientnet_b0":
            weights = _safe_weights(models.EfficientNet_B0_Weights, pretrained)
            m = models.efficientnet_b0(weights=weights)
            dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "efficientnet_b3":
            weights = _safe_weights(models.EfficientNet_B3_Weights, pretrained)
            m = models.efficientnet_b3(weights=weights)
            dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "efficientnet_b4":
            weights = _safe_weights(models.EfficientNet_B4_Weights, pretrained)
            m = models.efficientnet_b4(weights=weights)
            dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "efficientnet_v2_s":
            weights = _safe_weights(models.EfficientNet_V2_S_Weights, pretrained)
            m = models.efficientnet_v2_s(weights=weights)
            dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "convnext_tiny":
            weights = _safe_weights(models.ConvNeXt_Tiny_Weights, pretrained)
            m = models.convnext_tiny(weights=weights)
            dim = m.classifier[2].in_features
            m.classifier = nn.Identity()
            return m, dim

        if name == "resnet50":
            weights = _safe_weights(models.ResNet50_Weights, pretrained)
            m = models.resnet50(weights=weights)
            dim = m.fc.in_features
            m.fc = nn.Identity()
            return m, dim

    except Exception as e:
        print("Pretrained model loading failed, retrying with random weights.")
        print("Error:", repr(e))
        return create_backbone(name=name, pretrained=False)

    raise ValueError(f"Unsupported backbone: {name}")

class OAHybridNet(nn.Module):
    def __init__(self, backbone_name="efficientnet_b3", pretrained=True, num_classes=5, use_extra_se=True, dropout=0.35):
        super().__init__()
        self.num_classes = num_classes
        self.backbone, dim = create_backbone(backbone_name, pretrained=pretrained)
        self.feature_dim = dim
        self.use_extra_se = use_extra_se
        self.extra_se = SEBlock(dim, reduction=16) if use_extra_se else nn.Identity()
        self.dropout = nn.Dropout(dropout)

        self.cls_head = nn.Linear(dim, num_classes)

        # Ordered CORAL-style cutpoints:
        # logit_t = score - cutpoint_t, with cutpoint_0 < cutpoint_1 < ... guaranteed by softplus deltas.
        self.ordinal_score_head = nn.Linear(dim, 1)
        init_cutpoints = torch.linspace(-1.5, 1.5, steps=num_classes - 1)
        self.cutpoint_first = nn.Parameter(init_cutpoints[:1].clone())
        init_deltas = init_cutpoints[1:] - init_cutpoints[:-1]
        self.cutpoint_raw_deltas = nn.Parameter(torch.log(torch.expm1(init_deltas) + 1e-6))

    def ordered_cutpoints(self):
        if self.num_classes <= 2:
            return self.cutpoint_first
        deltas = F.softplus(self.cutpoint_raw_deltas)
        rest = self.cutpoint_first + torch.cumsum(deltas, dim=0)
        return torch.cat([self.cutpoint_first, rest], dim=0)

    def forward(self, x):
        feat = self.backbone(x)
        if feat.ndim > 2:
            feat = torch.flatten(feat, 1)
        feat = self.extra_se(feat)
        feat = self.dropout(feat)

        cls_logits = self.cls_head(feat)
        score = self.ordinal_score_head(feat)  # [B,1]
        cutpoints = self.ordered_cutpoints().view(1, -1)
        coral_logits = score - cutpoints
        ordinal_score = torch.sigmoid(coral_logits).sum(dim=1)

        return {
            "features": feat,
            "cls_logits": cls_logits,
            "coral_logits": coral_logits,
            "ordinal_score": ordinal_score,
            "cutpoints": cutpoints.detach(),
        }

def coral_logits_to_probs(coral_logits):
    # For K=5: P(y=0)=1-P(y>0), P(y=1)=P(y>0)-P(y>1), ..., P(y=4)=P(y>3)
    gt = torch.sigmoid(coral_logits)
    probs = torch.cat([
        1.0 - gt[:, 0:1],
        gt[:, 0:1] - gt[:, 1:2],
        gt[:, 1:2] - gt[:, 2:3],
        gt[:, 2:3] - gt[:, 3:4],
        gt[:, 3:4],
    ], dim=1)
    probs = probs.clamp(min=1e-7)
    probs = probs / probs.sum(dim=1, keepdim=True)
    return probs

def combined_probs(outputs, ordinal_blend=0.25):
    cls_probs = F.softmax(outputs["cls_logits"], dim=1)
    ord_probs = coral_logits_to_probs(outputs["coral_logits"])
    probs = (1.0 - ordinal_blend) * cls_probs + ordinal_blend * ord_probs
    probs = probs / probs.sum(dim=1, keepdim=True)
    return probs


# ---- verbatim from original notebook: 8) TRAIN / EVALUATE FUNCTIONS ----
def make_optimizer(model, config):
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("backbone."):
            backbone_params.append(p)
        else:
            head_params.append(p)
    return torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": config["LR_BACKBONE"]},
            {"params": head_params, "lr": config["LR_HEAD"]},
        ],
        weight_decay=config["WEIGHT_DECAY"]
    )

@torch.no_grad()
def run_inference(model, loader, config, use_tta_hflip=False):
    model.eval()
    all_y, all_idx, all_probs, all_cls_logits = [], [], [], []

    for x, y, idx in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        outputs = model(x)
        probs = combined_probs(outputs, ordinal_blend=config["ORDINAL_PROB_BLEND"])
        cls_logits = outputs["cls_logits"]

        if use_tta_hflip:
            xf = torch.flip(x, dims=[3])
            out_f = model(xf)
            probs_f = combined_probs(out_f, ordinal_blend=config["ORDINAL_PROB_BLEND"])
            cls_logits_f = out_f["cls_logits"]
            probs = 0.5 * (probs + probs_f)
            cls_logits = 0.5 * (cls_logits + cls_logits_f)

        all_y.append(y.cpu().numpy())
        all_idx.append(idx.numpy())
        all_probs.append(probs.cpu().numpy())
        all_cls_logits.append(cls_logits.cpu().numpy())

    y_true = np.concatenate(all_y)
    indices = np.concatenate(all_idx)
    probs = np.concatenate(all_probs)
    cls_logits = np.concatenate(all_cls_logits)
    y_pred = probs.argmax(axis=1)

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs,
        "cls_logits": cls_logits,
        "indices": indices,
        "metrics": compute_metrics(y_true, y_pred, probs=probs),
    }

def train_one_seed(config, seed=42, experiment_name="main"):
    set_seed(seed)
    out_dir = OUT_DIR / experiment_name / f"seed_{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "config_used.json", "w") as f:
        json.dump({k: (str(v) if isinstance(v, Path) else v) for k, v in config.items()}, f, indent=2)

    train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_loaders(config)
    train_counts = class_counts_from_dataset(train_ds)

    model = OAHybridNet(
        backbone_name=config["BACKBONE"],
        pretrained=config["PRETRAINED"],
        use_extra_se=config["USE_EXTRA_SE"],
        dropout=config.get("DROPOUT", 0.35),
    ).to(DEVICE)

    ce_criterion, ce_weight = build_ce_criterion(config, train_ds.labels)
    ce_criterion = ce_criterion.to(DEVICE)
    coral_pos_weight = threshold_pos_weights(train_ds.labels, num_classes=5).to(DEVICE)

    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["EPOCHS"], eta_min=1e-6
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(config["USE_AMP"] and DEVICE.type == "cuda"))

    best_score = -1e9
    best_state = None
    best_epoch = -1
    patience_left = config["PATIENCE"]
    history = []

    for epoch in range(1, config["EPOCHS"] + 1):
        model.train()
        t0 = time.time()
        running = defaultdict(float)
        n_batches = 0

        for x, y, idx in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(config["USE_AMP"] and DEVICE.type == "cuda")):
                outputs = model(x)
                loss, parts = hybrid_loss(outputs, y, ce_criterion, config, coral_pos_weight=coral_pos_weight)

            scaler.scale(loss).backward()
            if config["GRAD_CLIP"] is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), config["GRAD_CLIP"])
            scaler.step(optimizer)
            scaler.update()

            running["loss"] += float(loss.detach().cpu())
            for k, v in parts.items():
                running[k] += v
            n_batches += 1

        scheduler.step()

        val_out = run_inference(model, val_loader, config, use_tta_hflip=False)
        val_m = val_out["metrics"]
        score = val_selection_score(val_m, config)

        row = {
            "epoch": epoch,
            "seed": seed,
            "lr_backbone": optimizer.param_groups[0]["lr"],
            "lr_head": optimizer.param_groups[1]["lr"],
            "train_loss": running["loss"] / max(n_batches, 1),
            "train_loss_ce": running["loss_ce"] / max(n_batches, 1),
            "train_loss_coral": running["loss_coral"] / max(n_batches, 1),
            "train_loss_rank": running["loss_rank"] / max(n_batches, 1),
            "val_score": score,
            **{f"val_{k}": v for k, v in val_m.items()},
            "seconds": time.time() - t0,
        }
        history.append(row)

        improved = score > best_score
        if improved:
            best_score = score
            best_epoch = epoch
            # Store the best state on CPU to avoid accumulating large GPU copies across epochs/seeds.
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if config.get("SAVE_CHECKPOINTS", False):
                torch.save({
                    "model": best_state,
                    "config": config,
                    "seed": seed,
                    "epoch": epoch,
                    "val_metrics": val_m,
                    "val_score": score,
                }, out_dir / "best_model.pt")
            patience_left = config["PATIENCE"]
        else:
            patience_left -= 1

        print(
            f"[{experiment_name} seed={seed}] epoch {epoch:03d}/{config['EPOCHS']} "
            f"loss={row['train_loss']:.4f} val_acc={val_m['acc']:.4f} "
            f"val_qwk={val_m['qwk']:.4f} val_g1rec={val_m['g1_recall']:.4f} "
            f"score={score:.4f} {'*' if improved else ''}"
        )

        if patience_left <= 0:
            print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
            break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(out_dir / "history.csv", index=False)

    model.load_state_dict(best_state)

    val_final = run_inference(model, val_loader, config, use_tta_hflip=config["USE_TTA_HFLIP"])
    test_final = run_inference(model, test_loader, config, use_tta_hflip=config["USE_TTA_HFLIP"])

    # Save predictions
    for split_name, out in [("val", val_final), ("test", test_final)]:
        pred_df = pd.DataFrame({
            "idx": out["indices"],
            "y_true": out["y_true"],
            "y_pred": out["y_pred"],
        })
        for c in range(5):
            pred_df[f"prob_{c}"] = out["probs"][:, c]
            pred_df[f"cls_logit_{c}"] = out["cls_logits"][:, c]
        pred_df.to_csv(out_dir / f"{split_name}_predictions.csv", index=False)

        cm = confusion_matrix(out["y_true"], out["y_pred"], labels=np.arange(5))
        pd.DataFrame(cm).to_csv(out_dir / f"{split_name}_confusion_matrix.csv", index=False)
        per_class_table(out["y_true"], out["y_pred"]).to_csv(out_dir / f"{split_name}_per_class.csv", index=False)

    summary = {
        "experiment": experiment_name,
        "seed": seed,
        "best_epoch": best_epoch,
        **{f"val_{k}": v for k, v in val_final["metrics"].items()},
        **{f"test_{k}": v for k, v in test_final["metrics"].items()},
    }
    pd.DataFrame([summary]).to_csv(out_dir / "summary.csv", index=False)

    print_metrics(val_final["metrics"], title=f"VAL final seed={seed}")
    print_metrics(test_final["metrics"], title=f"TEST final seed={seed}")

    # Keep the returned model on CPU so multiple seeds do not occupy GPU memory.
    model.to("cpu")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "model": model,
        "summary": summary,
        "history": hist_df,
        "val": val_final,
        "test": test_final,
        "out_dir": out_dir,
    }


# =====================================================================
# REVISION ADDITIONS (BPEX-106665 major revision)
# Everything above this line is copied verbatim from the original
# training notebook that produced the published three-seed results, so
# the training protocol is unchanged. The code below only adds:
#   (1) an optional train/val exclusion filter (pHash-clean retraining),
#   (2) richer prediction export (file path, subject ID, side, separate
#       softmax / ordinal probabilities) for subject-level statistics,
#   (3) a run wrapper with metadata and a DONE marker for resumability.
# =====================================================================

import platform

# Base protocol: identical to config_used.json of the original main runs.
BASE_CONFIG = {
    "DATA_ROOT": "", "OUT_DIR": "",
    "IMG_SIZE": 380, "BATCH_SIZE": 8, "NUM_WORKERS": 2, "EPOCHS": 42, "PATIENCE": 10,
    "BACKBONE": "efficientnet_b4", "PRETRAINED": True, "USE_EXTRA_SE": True, "DROPOUT": 0.30,
    "USE_ROI_AUTOCROP": True,
    "LOSS_MODE": "weighted_ce", "LABEL_SMOOTHING": 0.02, "FOCAL_GAMMA": 1.0, "CB_BETA": 0.999,
    "CE_WEIGHT": 1.0, "ORDINAL_WEIGHT": 0.0, "RANK_WEIGHT": 0.0, "PAIRWISE_MARGIN": 1.0,
    "ORDINAL_PROB_BLEND": 0.0,
    "USE_WEIGHTED_SAMPLER": True, "SAMPLER_POWER": 0.25, "GRADE1_EXTRA_WEIGHT": 1.0,
    "VAL_SCORE_ACC_W": 0.85, "VAL_SCORE_QWK_W": 0.15, "VAL_SCORE_G1_W": 0.0,
    "USE_TTA_HFLIP": True,
    "LR_BACKBONE": 1.5e-5, "LR_HEAD": 1.5e-4, "WEIGHT_DECAY": 1e-4, "GRAD_CLIP": 1.0, "USE_AMP": True,
    "SAVE_CHECKPOINTS": True,
}

# Experimental arms. Only the keys listed here differ from BASE_CONFIG.
ARMS = {
    # 2x2 imbalance factorial (no ordinal terms)
    "rev_F_CE_nosampler":   {"LOSS_MODE": "ce",          "USE_WEIGHTED_SAMPLER": False},
    "rev_F_WCE_nosampler":  {"LOSS_MODE": "weighted_ce", "USE_WEIGHTED_SAMPLER": False},
    "rev_F_CE_sampler":     {"LOSS_MODE": "ce",          "USE_WEIGHTED_SAMPLER": True},
    "rev_F_WCE_sampler":    {"LOSS_MODE": "weighted_ce", "USE_WEIGHTED_SAMPLER": True},   # = original ce_baseline
    # moderate ordinal model (= original oa_ordnet_full)
    "rev_ORD_main":         {"ORDINAL_WEIGHT": 0.15, "RANK_WEIGHT": 0.08, "PAIRWISE_MARGIN": 1.0, "ORDINAL_PROB_BLEND": 0.10},
    # pHash-clean retraining of the two key models (train/val candidates removed, test unchanged)
    "rev_CLEAN_WCE_sampler": {"LOSS_MODE": "weighted_ce", "USE_WEIGHTED_SAMPLER": True, "_EXCLUDE_PHASH": True},
    "rev_CLEAN_ORD_main":    {"ORDINAL_WEIGHT": 0.15, "RANK_WEIGHT": 0.08, "PAIRWISE_MARGIN": 1.0, "ORDINAL_PROB_BLEND": 0.10, "_EXCLUDE_PHASH": True},
    # ordinal-weight sensitivity: blend and margin FIXED (0.10, 1.0) so that only lambda varies
    "rev_SENS_l005_r002":   {"ORDINAL_WEIGHT": 0.05, "RANK_WEIGHT": 0.02, "PAIRWISE_MARGIN": 1.0, "ORDINAL_PROB_BLEND": 0.10},
    "rev_SENS_l035_r020":   {"ORDINAL_WEIGHT": 0.35, "RANK_WEIGHT": 0.20, "PAIRWISE_MARGIN": 1.0, "ORDINAL_PROB_BLEND": 0.10},
}

# Mapping of already-completed original runs onto the revision arms (not retrained):
LEGACY_EQUIVALENTS = {
    ("rev_F_WCE_sampler", 42): "ce_baseline_efficientnet_b4_380/seed_42",
    ("rev_F_WCE_sampler", 7): "ce_baseline_efficientnet_b4_380/seed_7",
    ("rev_F_WCE_sampler", 123): "ce_baseline_efficientnet_b4_380/seed_123",
    ("rev_ORD_main", 42): "oa_ordnet_full_efficientnet_b4_380/seed_42",
    ("rev_ORD_main", 7): "oa_ordnet_full_efficientnet_b4_380/seed_7",
    ("rev_ORD_main", 123): "oa_ordnet_full_efficientnet_b4_380/seed_123",
    ("rev_F_CE_nosampler", 42): "ablation_SE_CE_no_sampler_efficientnet_b4_380_SE_ce_nosampler_ord0p0_rank0p0/seed_42",
}

# Strict pretrained guard: the original create_backbone silently falls back to random
# weights if the ImageNet download fails. For the revision runs this must fail loudly.
_original_create_backbone = create_backbone

def create_backbone(name="efficientnet_b3", pretrained=True):
    if pretrained and name.lower() == "efficientnet_b4":
        models.EfficientNet_B4_Weights.DEFAULT.get_state_dict(progress=False)  # raises if unavailable
    return _original_create_backbone(name=name, pretrained=pretrained)

EXCLUDE_FILENAMES = set()      # set per run; applied to train and val only, never to test
OUT_DIR = Path(".")
DATA_ROOT = None
SPLITS = None

_OriginalKneeOADataset = KneeOADataset

class KneeOADataset(_OriginalKneeOADataset):
    """Original dataset class plus an optional exclusion filter for non-test splits."""
    def __init__(self, split_dir, transform=None, allowed_labels=None):
        super().__init__(split_dir, transform=transform, allowed_labels=allowed_labels)
        self.n_excluded = 0
        is_test = SPLITS is not None and Path(split_dir).resolve() == Path(SPLITS["test"]).resolve()
        if EXCLUDE_FILENAMES and not is_test:
            kept = [(p, y) for p, y in self.samples if Path(p).name not in EXCLUDE_FILENAMES]
            self.n_excluded = len(self.samples) - len(kept)
            self.samples = kept
            self.labels = np.array([y for _, y in self.samples], dtype=np.int64)


def init(out_dir, data_root=""):
    global OUT_DIR, DATA_ROOT, SPLITS
    OUT_DIR = Path(out_dir)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    DATA_ROOT, SPLITS = find_data_root(data_root)
    return DATA_ROOT, SPLITS


def _subject_and_side(fname):
    m = re.match(r"^(\d{7})([LR])\.", fname)
    return (m.group(1), m.group(2)) if m else (None, None)


@torch.no_grad()
def extended_inference(model, loader, config):
    """Same forward pass and TTA as run_inference, but keeps softmax and ordinal
    probabilities separately so any inference blend can be reconstructed exactly."""
    model.eval()
    ds = loader.dataset
    rows_idx, ys, cls_p, ord_p, cls_l, osc = [], [], [], [], [], []
    for x, y, idx in loader:
        x = x.to(DEVICE, non_blocking=True)
        views = [x, torch.flip(x, dims=[3])] if config["USE_TTA_HFLIP"] else [x]
        cp, op, cl, sc = 0, 0, 0, 0
        for v in views:
            out = model(v)
            cp = cp + F.softmax(out["cls_logits"], dim=1)
            op = op + coral_logits_to_probs(out["coral_logits"])
            cl = cl + out["cls_logits"]
            sc = sc + out["ordinal_score"].view(-1)
        k = float(len(views))
        cls_p.append((cp / k).cpu().numpy()); ord_p.append((op / k).cpu().numpy())
        cls_l.append((cl / k).cpu().numpy()); osc.append((sc / k).cpu().numpy())
        ys.append(y.numpy()); rows_idx.append(idx.numpy())
    idx = np.concatenate(rows_idx); y = np.concatenate(ys)
    cls_p = np.concatenate(cls_p); ord_p = np.concatenate(ord_p)
    cls_l = np.concatenate(cls_l); osc = np.concatenate(osc)
    b = config["ORDINAL_PROB_BLEND"]
    comb = (1.0 - b) * cls_p + b * ord_p
    comb = comb / comb.sum(axis=1, keepdims=True)
    paths = [Path(ds.samples[i][0]) for i in idx]
    df = pd.DataFrame({
        "idx": idx,
        "relative_path": [p.relative_to(DATA_ROOT).as_posix() for p in paths],
        "filename": [p.name for p in paths],
    })
    ss = [_subject_and_side(p.name) for p in paths]
    df["subject_id"] = [s for s, _ in ss]
    df["side"] = [t for _, t in ss]
    df["y_true"] = y
    df["y_pred"] = comb.argmax(axis=1)
    for c in range(5):
        df[f"prob_{c}"] = comb[:, c]
    for c in range(5):
        df[f"cls_prob_{c}"] = cls_p[:, c]
    for c in range(5):
        df[f"ord_prob_{c}"] = ord_p[:, c]
    for c in range(5):
        df[f"cls_logit_{c}"] = cls_l[:, c]
    df["ordinal_score"] = osc
    return df.sort_values("idx").reset_index(drop=True)


def run_dir(exp_name, seed):
    return OUT_DIR / exp_name / f"seed_{seed}"


def is_done(exp_name, seed):
    return (run_dir(exp_name, seed) / "DONE.json").exists()


def run_one(exp_name, seed, exclude_filenames=None, epochs_override=None, patience_override=None):
    """Train one (arm, seed) with the original protocol and export extended outputs."""
    global EXCLUDE_FILENAMES
    arm = dict(ARMS[exp_name])
    use_excl = arm.pop("_EXCLUDE_PHASH", False)
    EXCLUDE_FILENAMES = set(exclude_filenames or []) if use_excl else set()

    cfg = copy.deepcopy(BASE_CONFIG)
    cfg.update(arm)
    cfg["OUT_DIR"] = str(OUT_DIR)
    if epochs_override is not None:
        cfg["EPOCHS"] = int(epochs_override)
    if patience_override is not None:
        cfg["PATIENCE"] = int(patience_override)

    t0 = time.time()
    res = train_one_seed(cfg, seed=seed, experiment_name=exp_name)
    out_dir = res["out_dir"]
    model = res["model"].to(DEVICE)

    train_ds, val_ds, test_ds, _, val_loader, test_loader = build_loaders(cfg)
    val_df = extended_inference(model, val_loader, cfg)
    test_df = extended_inference(model, test_loader, cfg)
    val_df.to_csv(out_dir / "val_predictions_ext.csv", index=False)
    test_df.to_csv(out_dir / "test_predictions_ext.csv", index=False)

    # Consistency check against the original export of train_one_seed.
    orig = res["test"]["probs"][np.argsort(res["test"]["indices"])]
    ext = test_df[[f"prob_{c}" for c in range(5)]].values
    max_abs_diff = float(np.abs(orig - ext).max())

    pd.Series(sorted(Path(p).name for p, _ in train_ds.samples)).to_csv(out_dir / "train_files_used.csv", index=False, header=["filename"])
    pd.Series(sorted(Path(p).name for p, _ in val_ds.samples)).to_csv(out_dir / "val_files_used.csv", index=False, header=["filename"])

    meta = {
        "experiment": exp_name, "seed": seed,
        "config_overrides": ARMS[exp_name],
        "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
        "n_train_excluded": int(getattr(train_ds, "n_excluded", 0)),
        "n_val_excluded": int(getattr(val_ds, "n_excluded", 0)),
        "train_class_counts": class_counts_from_dataset(train_ds).tolist(),
        "best_epoch": int(res["summary"]["best_epoch"]),
        "wall_time_hours": (time.time() - t0) / 3600.0,
        "ext_vs_original_test_prob_max_abs_diff": max_abs_diff,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "torch": torch.__version__, "python": platform.python_version(),
        "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    with open(out_dir / "run_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    with open(out_dir / "DONE.json", "w") as f:
        json.dump({"done": True, **{k: meta[k] for k in ["experiment", "seed", "wall_time_hours"]}}, f)

    del model, res
    EXCLUDE_FILENAMES = set()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return meta

In [ ]:
%%writefile worker.py
# worker.py — runs queued (arm, seed) jobs on one GPU. Generated by the revision notebook.
import argparse, json, os, sys, time, traceback
from pathlib import Path

ap = argparse.ArgumentParser()
ap.add_argument("--gpu", type=int, required=True)
ap.add_argument("--plan", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--data_root", default="")
ap.add_argument("--deadline", type=float, required=True)      # unix time after which no new run may finish
ap.add_argument("--est_hours", type=float, default=1.9)
ap.add_argument("--exclusion", required=True)
args = ap.parse_args()

os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)
sys.path.insert(0, str(Path(__file__).resolve().parent))
import koa_core as K   # noqa: E402

K.init(args.out, args.data_root)
plan = json.load(open(args.plan))
exclusion = json.load(open(args.exclusion))
exclusion_fn = [Path(p).name for p in exclusion]
status_path = Path(args.out) / f"_worker_gpu{args.gpu}_status.json"
est = args.est_hours
finished = []
stopped = False

def write_status(state, current=None):
    json.dump({"gpu": args.gpu, "state": state, "current": current, "finished": finished,
               "est_hours": est, "time": time.strftime("%H:%M:%S")}, open(status_path, "w"), indent=1)

for job in plan:
    exp, seed = job["experiment"], int(job["seed"])
    d = K.run_dir(exp, seed)
    if (d / "DONE.json").exists():
        continue
    if time.time() + est * 3600 > args.deadline:
        write_status("stopped_before_deadline"); stopped = True
        print(f"[gpu{args.gpu}] Not starting {exp}/seed_{seed}: would exceed session deadline.", flush=True)
        break
    d.mkdir(parents=True, exist_ok=True)
    lock = d / ".lock"
    try:
        fd = os.open(str(lock), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        os.write(fd, str(args.gpu).encode()); os.close(fd)
    except FileExistsError:
        continue  # another worker owns this job
    if (d / "DONE.json").exists():   # finished by the other worker between our check and the claim
        continue
    write_status("running", f"{exp}/seed_{seed}")
    print(f"[gpu{args.gpu}] START {exp} seed={seed}", flush=True)
    try:
        meta = K.run_one(exp, seed, exclude_filenames=exclusion_fn)
        finished.append({"run": f"{exp}/seed_{seed}", "hours": round(meta["wall_time_hours"], 2),
                         "check_diff": meta["ext_vs_original_test_prob_max_abs_diff"]})
        est = max(est, 1.15 * meta["wall_time_hours"])
        print(f"[gpu{args.gpu}] DONE {exp} seed={seed} in {meta['wall_time_hours']:.2f} h", flush=True)
    except Exception:
        (d / "ERROR.txt").write_text(traceback.format_exc())
        print(f"[gpu{args.gpu}] ERROR {exp} seed={seed}\n{traceback.format_exc()}", flush=True)
    # The lock is intentionally kept (also after errors) so that no job is run twice in one session;
    # the notebook removes stale locks at the start of the next session.
if not stopped:
    write_status("finished")
print(f"[gpu{args.gpu}] worker exit", flush=True)

In [ ]:
# pHash exclusion list: every train/val image that appears in a cross-split candidate pair (Hamming <= 5).
# Test images are never removed, so the full official test set is evaluated.
PHASH_EXCLUSION = ["train/0/9003430L.png", "train/0/9005075L.png", "train/0/9100442L.png", "train/0/9110292R.png", "train/0/9133340R.png", "train/0/9133694R.png", "train/0/9136484R.png", "train/0/9140040L.png", "train/0/9142466R.png", "train/0/9167589R.png", "train/0/9168944L.png", "train/0/9179589R.png", "train/0/9192585R.png", "train/0/9197546L.png", "train/0/9203516R.png", "train/0/9207016R.png", "train/0/9237473R.png", "train/0/9240548L.png", "train/0/9249027L.png", "train/0/9250574R.png", "train/0/9279874L.png", "train/0/9282240R.png", "train/0/9295223R.png", "train/0/9315064L.png", "train/0/9332151R.png", "train/0/9345249R.png", "train/0/9362450R.png", "train/0/9402055R.png", "train/0/9446883L.png", "train/0/9461885R.png", "train/0/9502364R.png", "train/0/9503396R.png", "train/0/9509373L.png", "train/0/9523742L.png", "train/0/9531421R.png", "train/0/9537961R.png", "train/0/9551668R.png", "train/0/9558648L.png", "train/0/9576572R.png", "train/0/9580736R.png", "train/0/9588488L.png", "train/0/9597417L.png", "train/0/9597417R.png", "train/0/9641901L.png", "train/0/9642894L.png", "train/0/9643032R.png", "train/0/9653659R.png", "train/0/9657071L.png", "train/0/9659659R.png", "train/0/9663968L.png", "train/0/9668982R.png", "train/0/9679705L.png", "train/0/9682706R.png", "train/0/9702644L.png", "train/0/9714513L.png", "train/0/9745242R.png", "train/0/9747379L.png", "train/0/9748587R.png", "train/0/9780165L.png", "train/0/9790287L.png", "train/0/9800154L.png", "train/0/9815467R.png", "train/0/9825282R.png", "train/0/9835291R.png", "train/0/9837774L.png", "train/0/9837774R.png", "train/0/9855117R.png", "train/0/9867760R.png", "train/0/9876530R.png", "train/0/9888515L.png", "train/0/9902576L.png", "train/0/9919499L.png", "train/0/9932578L.png", "train/0/9946473R.png", "train/0/9953518R.png", "train/0/9967932R.png", "train/0/9968036R.png", "train/0/9980800L.png", "train/0/9980800R.png", "train/1/9002316L.png", "train/1/9043894L.png", "train/1/9076127L.png", "train/1/9102297R.png", "train/1/9120519L.png", "train/1/9162906L.png", "train/1/9178581L.png", "train/1/9240515R.png", "train/1/9267508R.png", "train/1/9268233R.png", "train/1/9282182R.png", "train/1/9353703L.png", "train/1/9367389L.png", "train/1/9399980L.png", "train/1/9402367R.png", "train/1/9420706L.png", "train/1/9432327L.png", "train/1/9448563R.png", "train/1/9454360R.png", "train/1/9456105L.png", "train/1/9486466L.png", "train/1/9505417L.png", "train/1/9568618R.png", "train/1/9581241R.png", "train/1/9597688R.png", "train/1/9659659L.png", "train/1/9670760R.png", "train/1/9700163L.png", "train/1/9713904L.png", "train/1/9723575R.png", "train/1/9745458L.png", "train/1/9767446R.png", "train/1/9768869L.png", "train/1/9771214L.png", "train/1/9781154R.png", "train/1/9792535L.png", "train/1/9792535R.png", "train/1/9803881R.png", "train/1/9833782L.png", "train/1/9846831L.png", "train/1/9906027L.png", "train/1/9960949R.png", "train/2/9097409L.png", "train/2/9152196L.png", "train/2/9200014R.png", "train/2/9281052R.png", "train/2/9332058L.png", "train/2/9341535R.png", "train/2/9366809L.png", "train/2/9414488R.png", "train/2/9428112L.png", "train/2/9429319R.png", "train/2/9454360L.png", "train/2/9471214L.png", "train/2/9504363L.png", "train/2/9523947R.png", "train/2/9528513L.png", "train/2/9561686R.png", "train/2/9587941R.png", "train/2/9609230L.png", "train/2/9615420L.png", "train/2/9619698R.png", "train/2/9648234L.png", "train/2/9676751L.png", "train/2/9718167R.png", "train/2/9731208R.png", "train/2/9737820R.png", "train/2/9740824L.png", "train/2/9778518R.png", "train/2/9838272L.png", "train/2/9847107L.png", "train/2/9865818L.png", "train/2/9875231R.png", "train/2/9880321R.png", "train/2/9899098L.png", "train/2/9951312R.png", "train/2/9975327R.png", "train/2/9978026R.png", "train/3/9121285L.png", "train/3/9146462R.png", "train/3/9210045L.png", "train/3/9326831R.png", "train/3/9381251R.png", "train/3/9405154L.png", "train/3/9407052R.png", "train/3/9439411L.png", "train/3/9456548R.png", "train/3/9460377R.png", "train/3/9478732R.png", "train/3/9481413L.png", "train/3/9481413R.png", "train/3/9498365R.png", "train/3/9549107L.png", "train/3/9712762R.png", "train/3/9777033L.png", "train/3/9788072R.png", "train/3/9802320R.png", "train/3/9804361L.png", "train/3/9845361L.png", "train/3/9878594R.png", "train/3/9937239R.png", "train/4/9435335R.png", "train/4/9438852L.png", "val/0/9034812R.png", "val/0/9156526L.png", "val/0/9184108R.png", "val/0/9199070R.png", "val/0/9208055L.png", "val/0/9208055R.png", "val/0/9230504L.png", "val/0/9235549R.png", "val/0/9262046R.png", "val/0/9272393R.png", "val/0/9285207R.png", "val/0/9291210L.png", "val/0/9301641L.png", "val/0/9313233L.png", "val/0/9333793R.png", "val/0/9335175L.png", "val/0/9355228L.png", "val/0/9365192R.png", "val/0/9394122L.png", "val/0/9396054R.png", "val/0/9447878L.png", "val/0/9498724L.png", "val/0/9537182L.png", "val/0/9549868R.png", "val/0/9584925L.png", "val/0/9597737R.png", "val/0/9617748R.png", "val/0/9708353L.png", "val/0/9742066L.png", "val/0/9777471R.png", "val/0/9882107R.png", "val/0/9886598L.png", "val/0/9888738L.png", "val/0/9891187L.png", "val/0/9997856R.png", "val/0/9999365R.png", "val/1/9019907L.png", "val/1/9031961R.png", "val/1/9112495L.png", "val/1/9235549L.png", "val/1/9275309R.png", "val/1/9335752L.png", "val/1/9335752R.png", "val/1/9369225R.png", "val/1/9435315L.png", "val/1/9557111R.png", "val/1/9639581L.png", "val/1/9752895R.png", "val/1/9777471L.png", "val/1/9868067R.png", "val/1/9905383L.png", "val/2/9031961L.png", "val/2/9277151R.png", "val/2/9285207L.png", "val/2/9304509L.png", "val/2/9304509R.png", "val/2/9307242R.png", "val/2/9362978L.png", "val/2/9489653L.png", "val/2/9495960L.png", "val/2/9680344L.png", "val/2/9706468L.png", "val/2/9798591R.png", "val/2/9997869R.png", "val/3/9267648R.png", "val/3/9285212L.png", "val/3/9389580R.png", "val/3/9457264L.png", "val/3/9459371R.png", "val/3/9572718L.png", "val/3/9657464L.png", "val/3/9886598R.png"]
json.dump(PHASH_EXCLUSION, open(OUT_DIR / 'phash_exclusion_list.json', 'w'))
print(len(PHASH_EXCLUSION), 'excluded train/val images')

In [ ]:
# ======================= RUN PLAN (priority order) =======================
F_ARMS = ["rev_F_CE_nosampler", "rev_F_WCE_nosampler", "rev_F_CE_sampler", "rev_F_WCE_sampler"]
plan = []
def add(tier, exp, seed):
    if tier in ENABLED_TIERS:
        plan.append({"tier": tier, "experiment": exp, "seed": seed})

# Tier A: complete the factorial at the original seeds
for s in ORIGINAL_SEEDS:
    add("A", "rev_F_WCE_nosampler", s)
    add("A", "rev_F_CE_sampler", s)
    if s != 42:
        add("A", "rev_F_CE_nosampler", s)     # seed 42 exists from the original ablation
# Tier B: two additional seeds for all factorial cells and the ordinal model
for s in NEW_SEEDS:
    for a in F_ARMS + ["rev_ORD_main"]:
        add("B", a, s)
# Tier C: pHash-cleaned retraining of the two key models
for s in ORIGINAL_SEEDS:
    add("C", "rev_CLEAN_WCE_sampler", s)
    add("C", "rev_CLEAN_ORD_main", s)
# Tier D: lambda sensitivity with blend and margin fixed
for s in ORIGINAL_SEEDS:
    add("D", "rev_SENS_l005_r002", s)
    add("D", "rev_SENS_l035_r020", s)

json.dump(plan, open(OUT_DIR / "_plan.json", "w"), indent=1)
import pandas as pd
plan_df = pd.DataFrame(plan)
print(plan_df.groupby("tier").size().rename("runs").to_string())
print("Total planned new runs:", len(plan))

In [ ]:
# ======================= IMPORT COMPLETED RUNS FROM PREVIOUS SESSIONS =======================
imported = 0
if Path("/kaggle/input").exists():
    for done in Path("/kaggle/input").rglob("DONE.json"):
        seed_dir = done.parent
        exp_dir = seed_dir.parent
        if not exp_dir.name.startswith("rev_") or exp_dir.name.startswith("rev_SMOKE"):
            continue
        dst = OUT_DIR / exp_dir.name / seed_dir.name
        if (dst / "DONE.json").exists():
            continue
        shutil.copytree(seed_dir, dst, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("*.pt"))   # checkpoints stay in the earlier version output
        imported += 1
print("Imported completed runs:", imported)

# Remove stale locks / partial runs from an interrupted session
for lock in OUT_DIR.rglob(".lock"):
    lock.unlink()
def status_row(j):
    d = OUT_DIR / j["experiment"] / f"seed_{j['seed']}"
    return "done" if (d / "DONE.json").exists() else ("error" if (d / "ERROR.txt").exists() else "pending")
plan_df["status"] = [status_row(j) for j in plan]
print(plan_df.pivot_table(index="tier", columns="status", values="seed", aggfunc="count", fill_value=0))

In [ ]:
# ======================= PRE-FLIGHT CHECKS (CPU only) =======================
import importlib, re
sys.path.insert(0, str(Path.cwd()))
import koa_core as K
importlib.reload(K)
data_root, splits = K.init(OUT_DIR, DATA_ROOT)
print("DATA_ROOT:", data_root)

counts = {}
names = {}
for sp in ["train", "val", "test"]:
    ds = K.KneeOADataset(splits[sp], transform=None)
    counts[sp] = len(ds)
    names[sp] = [Path(p).name for p, _ in ds.samples]
print("Split sizes:", counts)
assert counts == {"train": 5778, "val": 826, "test": 1656}, "Unexpected split sizes"

# Subject-level disjointness from filename-encoded IDs (7-digit ID + L/R side)
subj = {sp: {n[:7] for n in names[sp]} for sp in names}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    print(f"subjects shared {a}-{b}:", len(subj[a] & subj[b]))
assert all(re.match(r"^\d{7}[LR]\.", n) for sp in names for n in names[sp]), "Unexpected filename pattern"

# Exclusion list must match existing train/val files exactly
all_nontest = set(names["train"]) | set(names["val"])
excl_fn = {Path(p).name for p in PHASH_EXCLUSION}
assert excl_fn <= all_nontest, "Some exclusion filenames were not found"
print("Exclusion: train", len(excl_fn & set(names["train"])), "| val", len(excl_fn & set(names["val"])))

# Download ImageNet weights once (prevents a download race between the two workers and
# fails loudly if internet is off, instead of silently training from random weights)
import torchvision
torchvision.models.EfficientNet_B4_Weights.DEFAULT.get_state_dict(progress=True)
print("ImageNet EfficientNet-B4 weights available.")

gpu_lines = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
N_GPUS = min(2, len(gpu_lines))
print("GPUs:", gpu_lines)
assert N_GPUS >= 1, "No GPU found. Enable a GPU accelerator."

In [ ]:
# ======================= SMOKE TEST (only if SMOKE_TEST=True) =======================
# One epoch of the pHash-clean ordinal arm: exercises training, ordinal losses, exclusion filter,
# extended export and the consistency check. Runs in a subprocess so the notebook holds no GPU memory.
if SMOKE_TEST:
    smoke_dir = WORK / "smoke_test"
    shutil.rmtree(smoke_dir, ignore_errors=True)
    code_str = (
        "import sys, json; sys.path.insert(0, '.'); import koa_core as K;"
        f"K.init(r'{smoke_dir}', r'{DATA_ROOT}');"
        f"excl=[p.split('/')[-1] for p in json.load(open(r'{OUT_DIR / 'phash_exclusion_list.json'}'))];"
        "m=K.run_one('rev_CLEAN_ORD_main', 0, exclude_filenames=excl, epochs_override=1, patience_override=1);"
        "print(json.dumps(m, indent=1))"
    )
    r = subprocess.run([sys.executable, "-c", code_str], capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-3000:])
    meta = json.load(open(smoke_dir / "rev_CLEAN_ORD_main" / "seed_0" / "run_meta.json"))
    ext = pd.read_csv(smoke_dir / "rev_CLEAN_ORD_main" / "seed_0" / "test_predictions_ext.csv")
    assert meta["n_train_excluded"] == 182 and meta["n_val_excluded"] == 72, meta
    assert meta["n_test"] == 1656 and len(ext) == 1656 and ext.subject_id.notna().all()
    assert meta["ext_vs_original_test_prob_max_abs_diff"] < 1e-3, meta["ext_vs_original_test_prob_max_abs_diff"]
    print("\nSMOKE TEST PASSED. Wall time (1 epoch): %.2f h" % meta["wall_time_hours"])
    shutil.rmtree(smoke_dir, ignore_errors=True)

In [ ]:
# ======================= LAUNCH WORKERS AND MONITOR =======================
if not SMOKE_TEST:
    deadline = SESSION_START + MAX_SESSION_HOURS * 3600
    procs = []
    for g in range(N_GPUS):
        log = open(OUT_DIR / f"_worker_gpu{g}.log", "a")
        cmd = [sys.executable, "worker.py", "--gpu", str(g), "--plan", str(OUT_DIR / "_plan.json"),
               "--out", str(OUT_DIR), "--data_root", DATA_ROOT, "--deadline", str(deadline),
               "--est_hours", str(EST_RUN_HOURS), "--exclusion", str(OUT_DIR / "phash_exclusion_list.json")]
        procs.append(subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT))
        time.sleep(90)   # stagger start-up
    def last_epoch_line(g):
        try:
            lines = (OUT_DIR / f"_worker_gpu{g}.log").read_text().splitlines()
            ep = [l for l in lines if "epoch" in l and l.startswith("[")]
            return ep[-1][:150] if ep else (lines[-1][:150] if lines else "")
        except Exception:
            return ""
    while any(p.poll() is None for p in procs):
        time.sleep(600)
        n_done = sum((OUT_DIR / j["experiment"] / f"seed_{j['seed']}" / "DONE.json").exists() for j in plan)
        print(f"[{(time.time()-SESSION_START)/3600:5.2f} h] done {n_done}/{len(plan)}")
        for g in range(N_GPUS):
            print(f"   gpu{g}: {last_epoch_line(g)}")
    print("Workers finished. Exit codes:", [p.returncode for p in procs])

In [ ]:
# ======================= COLLECT RESULTS =======================
rows = []
for meta_path in sorted(OUT_DIR.glob("rev_*/seed_*/run_meta.json")):
    d = meta_path.parent
    if not (d / "DONE.json").exists():
        continue
    meta = json.load(open(meta_path))
    s = pd.read_csv(d / "summary.csv").iloc[0].to_dict()
    rows.append({**{k: meta[k] for k in ["experiment", "seed", "n_train", "n_val", "best_epoch",
                                          "wall_time_hours", "ext_vs_original_test_prob_max_abs_diff"]},
                 **{k: s[k] for k in ["val_acc", "val_qwk", "test_acc", "test_qwk", "test_mae",
                                      "test_macro_f1", "test_g1_recall", "test_g1_precision"]}})
res_df = pd.DataFrame(rows)
res_df.to_csv(OUT_DIR / "revision_runs_summary.csv", index=False)
if len(res_df):
    print(res_df.groupby("experiment")[["test_acc", "test_qwk", "test_macro_f1", "test_g1_recall"]]
          .agg(["count", "mean", "std"]).round(4).to_string())
errors = list(OUT_DIR.rglob("ERROR.txt"))
print("Runs with errors:", [str(e.parent.relative_to(OUT_DIR)) for e in errors])

# Zip everything except model checkpoints for download / analysis
zip_base = WORK / "koa_revision_results"
tmp = WORK / "_zip_staging"
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(OUT_DIR, tmp / OUT_DIR.name, ignore=shutil.ignore_patterns("*.pt", ".lock"))
shutil.make_archive(str(zip_base), "zip", tmp)
shutil.rmtree(tmp, ignore_errors=True)
print("Wrote", zip_base.with_suffix(".zip"), f"({zip_base.with_suffix('.zip').stat().st_size/1e6:.1f} MB)")